In [0]:
%sql
USE CATALOG workspace;
USE SCHEMA notebook_breweries;

In [0]:
df = spark.read.table("notebook_breweries.silver_staging_breweries")
display(df)

###Snapshot Completo

In [0]:
%skip 
spark.sql("DROP TABLE IF EXISTS bronze_breweries")

In [0]:
from pyspark.sql import functions as F

(
    df
    .select(
            'address_1', 
            'address_2', 
            'address_3', 
            'brewery_type', 
            'city', 
            'country', 
            'id', 
            'latitude', 
            'longitude', 
            'name', 
            'phone', 
            'postal_code', 
            'state', 
            'state_province', 
            'street', 
            'website_url',
            'ingestion_ts'
    )
    .where(
        (F.col("country") == "United States") &
        (F.col("address_1").isNotNull())         
    )
    .withColumn(
        "address_2",
        F.when(
            F.col("address_2").isNull(),
            F.lit("Doesn't exist")
        ).otherwise( 
            F.col("address_2")
        )
    )
    .withColumn(
        "address_3",
        F.when(
            F.col("address_3").isNull(),
            F.lit("Doesn't exist")
        ).otherwise( 
            F.col("address_3")
        )
    )
    .withColumn(
        "name",
        F.regexp_replace(F.col("name"), "Â", "")
    )
    .withColumn(
        "phone",
        F.when(
            F.col('phone').isNull(), 
            F.lit("Unknown")
        ).otherwise(
            F.regexp_replace((F.col('phone')), r'^\+\d{1,3}\s', '')
        )
    )
    .withColumn(
        "phone",
        F.regexp_replace(F.col("phone"), r"[^\d]", "") 
    )
    .withColumn(
        "postal_code",
        F.when(
             F.col("postal_code").isNull(), 
             F.lit("Unknown")
        )
        .when(
            F.col("country") == "United States",
            F.regexp_replace(F.col("postal_code"), r"-.*", "")
        )
        .otherwise(
            F.col("postal_code")
        )
    )
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "True")
    .saveAsTable("silver_breweries")

)

###Test per SCD2

In [0]:
%skip
from pyspark.sql import functions as F

silver_test = (
    spark.read.table("silver_breweries")
        .withColumn(
            "street",
            F.when(
                F.col("id") == "4dcaeaa3-d7cc-4016-9392-5bde4e3a8f4d",
                F.lit("39 River Rd Street 6")
            ).otherwise(F.col("street") )
        )
        .withColumn(
            "ingestion_ts",
            F.lit(run_ts).cast("timestamp")
    )
)

In [0]:
%skip
(
    silver_test
            .write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema","true")
            .saveAsTable("silver_breweries")
)

In [0]:
%skip
selezione = spark.sql("""
SELECT id, name, street, valid_from, valid_to, current
FROM gold_breweries
WHERE id = '4dcaeaa3-d7cc-4016-9392-5bde4e3a8f4d'
ORDER BY valid_from
""")

display(selezione)

In [0]:
%skip
spark.sql("""
    SELECT id
    FROM gold_breweries
    GROUP BY id
    HAVING COUNT_IF(current = true) > 1
""")

In [0]:
%skip
display(
    spark.sql(
        """
        SELECT *
        FROM gold_breweries
        WHERE valid_to IS NOT NULL
        AND valid_to < valid_from
        """
        ) 
)

In [0]:
%skip
silver_test = (
                silver_test
                        .withColumn(
                            "ingestion_ts",
                            F.current_timestamp()
                        )
        )

In [0]:
%skip
row_selection = spark.sql("SELECT * FROM silver_breweries")
display(row_selection)